In [4]:
import numpy as np
import math
import itertools
import plotly.graph_objects as go

In [ ]:
# generate all counts

def make_counts(dice):
    dice = np.array(dice)
    count = []
    count.append(np.sum(dice == 1))
    count.append(np.sum(dice == 2))
    count.append(np.sum(dice == 3))
    count.append(np.sum(dice == 4))
    count.append(np.sum(dice == 5))
    count.append(np.sum(dice == 6))
    return count

roll5 = list(itertools.combinations_with_replacement(range(1,7), 5))
roll4 = list(itertools.combinations_with_replacement(range(1,7), 4))
roll3 = list(itertools.combinations_with_replacement(range(1,7), 3))
roll2 = list(itertools.combinations_with_replacement(range(1,7), 2))
roll1 = np.array([1,2,3,4,5,6])


roll5 = [make_counts(dice) for dice in roll5]
roll4 = [make_counts(dice) for dice in roll4]
roll3 = [make_counts(dice) for dice in roll3]
roll2 = [make_counts(dice) for dice in roll2]
roll1 = [make_counts(dice) for dice in roll1]


DELTA_CACHE = {0: [0,0,0,0,0,0],
               1: roll1,
               2: roll2,
               3: roll3,
               4: roll4,
               5: roll5
}

state_index = {tuple(s): i for i, s in enumerate(roll5)}
print(state_index)

{(5, 0, 0, 0, 0, 0): 0, (4, 1, 0, 0, 0, 0): 1, (4, 0, 1, 0, 0, 0): 2, (4, 0, 0, 1, 0, 0): 3, (4, 0, 0, 0, 1, 0): 4, (4, 0, 0, 0, 0, 1): 5, (3, 2, 0, 0, 0, 0): 6, (3, 1, 1, 0, 0, 0): 7, (3, 1, 0, 1, 0, 0): 8, (3, 1, 0, 0, 1, 0): 9, (3, 1, 0, 0, 0, 1): 10, (3, 0, 2, 0, 0, 0): 11, (3, 0, 1, 1, 0, 0): 12, (3, 0, 1, 0, 1, 0): 13, (3, 0, 1, 0, 0, 1): 14, (3, 0, 0, 2, 0, 0): 15, (3, 0, 0, 1, 1, 0): 16, (3, 0, 0, 1, 0, 1): 17, (3, 0, 0, 0, 2, 0): 18, (3, 0, 0, 0, 1, 1): 19, (3, 0, 0, 0, 0, 2): 20, (2, 3, 0, 0, 0, 0): 21, (2, 2, 1, 0, 0, 0): 22, (2, 2, 0, 1, 0, 0): 23, (2, 2, 0, 0, 1, 0): 24, (2, 2, 0, 0, 0, 1): 25, (2, 1, 2, 0, 0, 0): 26, (2, 1, 1, 1, 0, 0): 27, (2, 1, 1, 0, 1, 0): 28, (2, 1, 1, 0, 0, 1): 29, (2, 1, 0, 2, 0, 0): 30, (2, 1, 0, 1, 1, 0): 31, (2, 1, 0, 1, 0, 1): 32, (2, 1, 0, 0, 2, 0): 33, (2, 1, 0, 0, 1, 1): 34, (2, 1, 0, 0, 0, 2): 35, (2, 0, 3, 0, 0, 0): 36, (2, 0, 2, 1, 0, 0): 37, (2, 0, 2, 0, 1, 0): 38, (2, 0, 2, 0, 0, 1): 39, (2, 0, 1, 2, 0, 0): 40, (2, 0, 1, 1, 1, 0): 41, (

In [3]:
# functions that score a count

def calc_ones(count):
    return count[0]

def calc_twos(count):
    return count[1]

def calc_threes(count):
    return count[2]
    
def calc_fours(count):
    return count[3]

def calc_fives(count):
    return count[4]
    
def calc_sixes(count):
    return count[5]

def get_score_chance(count):
    score = 0
    for i in range(6):
        score += count[i] * (i + 1)
    return score

def calc_three_of_a_kind(count):
    count = np.array(count)
    threes_mask = count[count >= 3]
    if len(threes_mask) != 0:
        score = 0
        for i in range(6):
            score += count[i] * (i + 1)
        return score
    else:
        return 0

def calc_four_of_a_kind(count):
    count = np.array(count)
    threes_mask = count[count >= 4]
    if len(threes_mask) != 0:
        score = 0
        for i in range(6):
            score += count[i] * (i + 1)
        return score
    else:
        return 0

def full_house(count):
    count = np.array(count)
    counts_mask = count[count > 0]
    if len(counts_mask) == 2:
        if counts_mask[0] == 2 or counts_mask[0] == 3:
            return 25
        else:
            return 0
    else:
        return 0
    
def small_straight(count):
    count = np.array(count)
    for start in range(3):
        if np.all(count[start:start+4] >= 1):
            return 30
    return 0

def large_straight(count):
    count = np.array(count)
    for start in range(2):
        if np.all(count[start:start+5] >= 1):
            return 40
    return 0

def yahtzee(count):
    for i in count:
        if i == 5:
            return 50     
    return 0

In [ ]:

# Code for the figures in 4.1

def multinomial_prob(count):
    r = sum(count)  # should be 5
    numer = math.factorial(r)

    denom = 1
    for c in count:
        denom *= math.factorial(c)

    return numer / denom * (1/6)**r

def make_histogram(probs, category):
    fig = go.Figure()


    fig.add_trace(
        go.Bar(
            x=np.arange(len(probs)),   # indices as categories
            y=probs,                   # probabilities
        )
    )

    
    fig.update_layout(
        title=f"{category} PMF",
        xaxis_title="Score",
        yaxis_title="Probability",
        bargap=0.1
    )

    fig.show()


def compute_PMF(category, function, multinomial_prob, make_histogram, counts):
    scores = []
    probabilities = []
    for count in counts:
        probabilities.append(multinomial_prob(count))
        scores.append(function(count))

    pmf = np.zeros(50)

    for i in range(252):
        pmf[scores[i] * 4] += probabilities[i]

    make_histogram(pmf, category)

    return pmf, probabilities



# test1 = compute_PMF('chance', get_score_chance, multinomial_prob, make_histogram, roll5)
# test2, probabilities = compute_PMF('three_of_a_kind', calc_three_of_a_kind, multinomial_prob, make_histogram, roll5)
# test3 = compute_PMF('full_house', full_house, multinomial_prob, make_histogram, roll5)
test4 = compute_PMF('fours', calc_fours, multinomial_prob, make_histogram, roll5)
    

In [ ]:
# Multinomial functions
def multinomial_prob(delta, r):
    numer = math.factorial(r)
    denom = 1
    for d in delta:
        denom *= math.factorial(int(d))
    return numer / denom * (1/6)**r

def all_keeps_for_state(counts):
    c1,c2,c3,c4,c5,c6 = counts
    keeps = []
    for k1 in range(c1+1):
        for k2 in range(c2+1):
            for k3 in range(c3+1):
                for k4 in range(c4+1):
                    for k5 in range(c5+1):
                        for k6 in range(c6+1):
                            keeps.append((k1,k2,k3,k4,k5,k6))
    return keeps 

In [ ]:
from collections import defaultdict

# DYNAMIC PROGRAMMING

def compute_pmf_for_state(score_fn, max_rolls, STATES, STATE_INDEX, 
                          DELTA_CACHE, all_keeps_for_state):
    
    n_states = len(STATES)
    # DP table only for reachable states
    F = {}
    keep_dict = {}

    # Base case r = 0
    for s_idx, c in enumerate(STATES):
        F[(s_idx, 0)] = {score_fn(c): 1.0}
    
    # Fill DP for r = 1,2
    for r in range(1, max_rolls + 1):
        for s_idx, c in enumerate(STATES):
            best_E = None
            best_pmf = None
            best_keep = None

            for keep in all_keeps_for_state(tuple(c)):
                kept_total = sum(keep)
                r_dice = 5 - kept_total

                if r_dice == 0:
                    pmf = F[(s_idx, r-1)]

                else:
                    pmf_temp = defaultdict(float)

                    for delta in DELTA_CACHE[r_dice]:
                        p_delta = multinomial_prob(delta, r_dice)
                        new_counts = tuple(ki + di for ki, di in zip(keep, delta))
                        new_idx = STATE_INDEX[new_counts]

                        next_pmf = F[(new_idx, r-1)]

                        for sc, p_sc in next_pmf.items():
                            pmf_temp[sc] += p_delta * p_sc

                    pmf = dict(pmf_temp)

                # expected value
                E = sum(sc * p for sc, p in pmf.items())
                if best_E is None or E > best_E:
                    best_E = E
                    best_pmf = pmf
                    best_keep = keep

            best_pmf = {sc: round(p, 3) for sc, p in best_pmf.items()}
            F[(s_idx, r)] = best_pmf
            keep_dict[(s_idx, r)] = tuple(best_keep)

    # I dont use the keep_dict, ignore that
    return F, keep_dict

In [ ]:
import time, pickle

# COMPOSE DP STATES V(c,r) for each category and save

yaht_dict = {}
three_of_a_dict = {}
four_of_a_dict = {}
full_dict = {}
sm_str_dict = {}
lrg_str_dict = {}
chance_dict = {}
ones_dict = {}
twos_dict = {}
threes_dict = {}
fours_dict = {}
fives_dict = {}
sixes_dict = {}


F1 = compute_pmf_for_state( yahtzee, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F2 = compute_pmf_for_state( calc_three_of_a_kind, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F3 = compute_pmf_for_state( calc_four_of_a_kind, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F4 = compute_pmf_for_state( full_house, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F5 = compute_pmf_for_state( small_straight, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F6 = compute_pmf_for_state( large_straight, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F7 = compute_pmf_for_state( get_score_chance, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F8 = compute_pmf_for_state( calc_ones, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F9 = compute_pmf_for_state( calc_twos, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F10 = compute_pmf_for_state( calc_threes, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F11 = compute_pmf_for_state( calc_fours, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F12 = compute_pmf_for_state( calc_fives, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F13 = compute_pmf_for_state( calc_sixes, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)

def pickle_it(the_dict, name):
    with open(f"all_pmfs/{name}.pkl", "wb") as f:
        pickle.dump(the_dict, f)

pickle_it(F1, 'yahtzee__DP_2')
pickle_it(F2, 'three_of_a_kind_DP_2')
pickle_it(F3, 'four_of_a_kind_DP_2')
pickle_it(F4, 'full_house_DP_2')
pickle_it(F5, 'small_straight_DP_2')
pickle_it(F6, 'large_straight_DP_2')
pickle_it(F7, 'chance_DP_2')
pickle_it(F8, 'ones_DP_2')
pickle_it(F9, 'twos_DP_2')
pickle_it(F10, 'threes_DP_2')
pickle_it(F11, 'fours_DP_2')
pickle_it(F12, 'fives_DP_2')
pickle_it(F13, 'sixes_DP_2')

NameError: name 'compute_pmf_for_state' is not defined

In [ ]:
from collections import defaultdict

# GREEDY FUNCTION

def compute_pmf_for_state_greedy(score_fn, max_rolls, STATES, STATE_INDEX, 
                                     DELTA_CACHE, all_keeps_for_state):
    F = {}
    keep_dict = {}

    # Base case r = 0
    for s_idx, c in enumerate(STATES):
        F[(s_idx, 0)] = {score_fn(c): 1.0}

    # r = 1,2
    for r in range(1, max_rolls + 1):
        for s_idx, c in enumerate(STATES):
            best_obj = None 
            best_pmf = None
            best_keep = None

            for keep in all_keeps_for_state(tuple(c)):
                kept_total = sum(keep)
                r_dice = 5 - kept_total

                if r_dice == 0:
                    pmf = F[(s_idx, r-1)]

                else:
                    pmf_temp = defaultdict(float)

                    for delta in DELTA_CACHE[r_dice]:
                        p_delta = multinomial_prob(delta, r_dice)
                        new_counts = tuple(ki + di for ki, di in zip(keep, delta))
                        new_idx = STATE_INDEX[new_counts]

                        next_pmf = F[(new_idx, r-1)]
                        for sc, p_sc in next_pmf.items():
                            pmf_temp[sc] += p_delta * p_sc

                    pmf = dict(pmf_temp)

                # Greedy objective takes highest possible score in this PMF instead
                greedy_score = max(pmf.keys())

                if best_obj is None or greedy_score > best_obj:
                    best_obj = greedy_score
                    best_pmf = pmf
                    best_keep = keep

            best_pmf = {sc: round(p, 3) for sc, p in best_pmf.items()}

            F[(s_idx, r)] = best_pmf
            keep_dict[(s_idx, r)] = tuple(best_keep)

    # again ignore the keep_dict 
    return F, keep_dict


In [ ]:
import time, pickle

# make same dictionary for greedy so that it can be run on Yahtzee.ipynb

yaht_dict = {}
three_of_a_dict = {}
four_of_a_dict = {}
full_dict = {}
sm_str_dict = {}
lrg_str_dict = {}
chance_dict = {}
ones_dict = {}
twos_dict = {}
threes_dict = {}
fours_dict = {}
fives_dict = {}
sixes_dict = {}

F1 = compute_pmf_for_state_greedy(yahtzee, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F2 = compute_pmf_for_state_greedy(calc_three_of_a_kind, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F3 = compute_pmf_for_state_greedy(calc_four_of_a_kind, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F4 = compute_pmf_for_state_greedy(full_house, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F5 = compute_pmf_for_state_greedy(small_straight, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F6 = compute_pmf_for_state_greedy(large_straight, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F7 = compute_pmf_for_state_greedy(get_score_chance, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F8 = compute_pmf_for_state_greedy(calc_ones, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F9 = compute_pmf_for_state_greedy(calc_twos, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F10 = compute_pmf_for_state_greedy(calc_threes, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F11 = compute_pmf_for_state_greedy(calc_fours, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F12 = compute_pmf_for_state_greedy(calc_fives, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)
F13 = compute_pmf_for_state_greedy(calc_sixes, 2, roll5, state_index, DELTA_CACHE, all_keeps_for_state)

def pickle_it(the_dict, name):
    with open(f"all_greedy/{name}.pkl", "wb") as f:
        pickle.dump(the_dict, f)

pickle_it(F1, 'yahtzee_greedy2')
pickle_it(F2, 'three_of_a_kind_greedy2')
pickle_it(F3, 'four_of_a_kind_greedy2')
pickle_it(F4, 'full_house_greedy2')
pickle_it(F5, 'small_straight_greedy2')
pickle_it(F6, 'large_straight_greedy2')
pickle_it(F7, 'chance_greedy2')
pickle_it(F8, 'ones_greedy2')
pickle_it(F9, 'twos_greedy2')
pickle_it(F10, 'threes_greedy2')
pickle_it(F11, 'fours_greedy2')
pickle_it(F12, 'fives_greedy2')
pickle_it(F13, 'sixes_greedy2')